# 4.1 Local search: hill climbing and simulated annealing: practice notebook

When the space is too big to search and only the final state matters, keep one state and improve it. Hill climbing is fast but gets stuck. Restarts, beam search and simulated annealing help it escape.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### The 8 queens objective: attacking pairs

In [ ]:
# The 8 queens objective: h = number of pairs of queens that attack each other.
# board[c] is the row of the queen in column c (one queen per column).
def attacking_pairs(board):
    count = 0
    n = len(board)
    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            same_row = board[c1] == board[c2]
            same_diagonal = abs(board[c1] - board[c2]) == c2 - c1
            if same_row or same_diagonal:
                count = count + 1
    return count

print("h =", attacking_pairs([4, 5, 6, 3, 4, 5, 6, 5]))   # the textbook start state, h = 17
print("h =", attacking_pairs([0, 4, 7, 5, 2, 6, 1, 3]))   # a solution, h = 0

### Hill climbing against simulated annealing on a small landscape

In [ ]:
# Hill climbing and simulated annealing on a small landscape. value[x] is the height of state x.
# Neighbours of x are x - 1 and x + 1. We want the highest point.
import math
import random

value = [1, 3, 5, 4, 4, 6, 7, 9, 6, 3]

def hill_climb(x):
    while True:
        best = x
        for n in [x - 1, x + 1]:
            if 0 <= n < len(value) and value[n] > value[best]:
                best = n
        if best == x:
            return x                      # no neighbour is higher: a peak (maybe only a local one)
        x = best

def anneal(x, T, alpha, rounds, seed):
    random.seed(seed)
    for k in range(rounds):
        n = x + random.choice([-1, 1])
        if n < 0 or n >= len(value):
            continue
        delta = value[n] - value[x]
        if delta > 0 or random.random() < math.exp(delta / T):
            x = n                         # always go up, sometimes go down
        T = T * alpha                     # geometric cooling
    return x

print("hill climbing from 1 stops at", hill_climb(1), "with height", value[hill_climb(1)])
x = anneal(1, 5.0, 0.85, 30, 4)
print("annealing from 1 ends at", x, "with height", value[x])
# Annealing is random. Over seeds 0 to 199 it reaches the top (x = 7) in 90 runs. Hill climbing from 1 never does.

## Your turn

### Task 1: Count attacking queens

Write `attacking_pairs(board)` where board[c] is the row of the queen in column c. Return the number of pairs of queens in the same row or on the same diagonal.

Hint: Two queens attack on a diagonal when the row difference equals the column difference.

In [ ]:
def attacking_pairs(board):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('attacking_pairs([4, 5, 6, 3, 4, 5, 6, 5])', 17), ('attacking_pairs([0, 4, 7, 5, 2, 6, 1, 3])', 0), ('attacking_pairs([0, 0, 0, 0])', 6), ('attacking_pairs([0, 1, 2, 3])', 6)])

### Task 2: Steepest descent on 8 queens

Write `hill_climb(board)` that repeats: try every move of one queen to another row in its column, and take the move with the fewest attacking pairs (the first such move in column then row order) if it is strictly better than now. Stop when no move is better. Return [final board, final number of attacking pairs]. A helper attacking_pairs is given.

Hint: Copy the board for each trial move. Keep the best trial only if it is strictly better.

In [ ]:
def attacking_pairs(board):
    count = 0
    for c1 in range(len(board)):
        for c2 in range(c1 + 1, len(board)):
            if board[c1] == board[c2] or abs(board[c1] - board[c2]) == c2 - c1:
                count = count + 1
    return count

def hill_climb(board):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('hill_climb([0, 4, 7, 5, 2, 6, 1, 3])', [[0, 4, 7, 5, 2, 6, 1, 3], 0]), ('hill_climb([1, 3, 0, 2])[1]', 0), ('hill_climb([0, 0, 0, 0])[1] < 6', True), ('hill_climb([4, 5, 6, 3, 4, 5, 6, 5])[1] < 17', True)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Count attacking queens

The textbook start state has h = 17.

In [ ]:
def attacking_pairs(board):
    count = 0
    for c1 in range(len(board)):
        for c2 in range(c1 + 1, len(board)):
            if board[c1] == board[c2] or abs(board[c1] - board[c2]) == c2 - c1:
                count = count + 1
    return count

In [ ]:
check([('attacking_pairs([4, 5, 6, 3, 4, 5, 6, 5])', 17), ('attacking_pairs([0, 4, 7, 5, 2, 6, 1, 3])', 0), ('attacking_pairs([0, 0, 0, 0])', 6), ('attacking_pairs([0, 1, 2, 3])', 6)])

### Solution to task 2: Steepest descent on 8 queens

It always ends at a local minimum, which is a solution only sometimes.

In [ ]:
def attacking_pairs(board):
    count = 0
    for c1 in range(len(board)):
        for c2 in range(c1 + 1, len(board)):
            if board[c1] == board[c2] or abs(board[c1] - board[c2]) == c2 - c1:
                count = count + 1
    return count

def hill_climb(board):
    board = board[:]
    current = attacking_pairs(board)
    while True:
        best_board = None
        best_value = current
        for col in range(len(board)):
            for row in range(len(board)):
                if row != board[col]:
                    trial = board[:]
                    trial[col] = row
                    value = attacking_pairs(trial)
                    if value < best_value:
                        best_value = value
                        best_board = trial
        if best_board is None:
            return [board, current]
        board = best_board
        current = best_value

In [ ]:
check([('hill_climb([0, 4, 7, 5, 2, 6, 1, 3])', [[0, 4, 7, 5, 2, 6, 1, 3], 0]), ('hill_climb([1, 3, 0, 2])[1]', 0), ('hill_climb([0, 0, 0, 0])[1] < 6', True), ('hill_climb([4, 5, 6, 3, 4, 5, 6, 5])[1] < 17', True)])